# Training-scene class diagnostic
Attach BTP-code-training-diagnostic.zip, quarter_lr_to_epoch100_20260918_094320.zip, and prepared public252 data. Enable T4 and run in order. This is inference only on eight class/scene selections, with no optimizer updates. Full-image and matched ROI/crop results are explicitly separated. Download the final ZIP.

In [ ]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-training-diagnostic')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
# Identify this evaluator by its actual feature, not a shared filename.
import tempfile
marker = 'Training-only inference diagnostic; fixed class-rich crops'
def is_audit_source(path):
    return path.is_file() and marker in path.read_text()
if not is_audit_source(CODE / 'diagnose_training_classes.py'):
    sources = [p.parent for p in INPUTS.rglob('diagnose_training_classes.py') if is_audit_source(p)]
    if sources:
        shutil.copytree(sorted(sources)[0], CODE, dirs_exist_ok=True)
    else:
        matches = []
        archives = list(INPUTS.rglob('*.zip'))
        for archive in archives:
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                for member in z.namelist():
                    if member.endswith('/diagnose_training_classes.py') and marker.encode() in z.read(member):
                        matches.append((archive, member))
        if not matches:
            print('ZIP inputs found:', [str(p) for p in archives])
            raise FileNotFoundError('Updated audit code is not attached. Add BTP-code-training-diagnostic.zip, then rerun this cell.')
        archive, member = sorted(matches, key=lambda x: str(x[0]))[0]
        print('Using audit code:', archive)
        with tempfile.TemporaryDirectory(prefix='weak_audit_code_', dir='/kaggle/working') as tmp:
            staging = Path(tmp)
            with zipfile.ZipFile(archive) as z: safe_extract(z, staging)
            shutil.copytree((staging / member).parent, CODE, dirs_exist_ok=True)
assert is_audit_source(CODE / 'diagnose_training_classes.py'), 'Audit evaluator not found after extraction.'
import json, hashlib, math
import torch
expected = 'quarter_lr_to_epoch100_20260918_094320'
relative = expected + '/model/last.pt'
refs = list(INPUTS.rglob(relative))
if not refs:
    working = Path('/kaggle/working') / relative
    if working.is_file(): refs = [working]
if not refs:
    dest = Path('/kaggle/working/restored_quarter60')
    restored = dest / relative
    if restored.is_file(): refs = [restored]
    else:
        for archive in INPUTS.rglob('*.zip'):
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                if relative in z.namelist():
                    safe_extract(z, dest)
                    refs = [dest / relative]
                    break
assert len(refs) == 1, 'Attach quarter_lr_to_epoch100_20260918_094320.zip or its extracted folder.'
CHECKPOINT = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert CHECKPOINT.with_name(name).is_file(), f'Missing {name}'
CHECKPOINT = CHECKPOINT.with_name('best_iou.pth')
assert (CODE / 'diagnose_training_classes.py').is_file()
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Evaluating saved best segmentation checkpoint:', CHECKPOINT)


In [ ]:
from datetime import datetime
from IPython.display import display, FileLink
NAME = 'training_class_diagnostic_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = Path('/kaggle/working') / NAME
try:
    subprocess.run([sys.executable, '-u', 'diagnose_training_classes.py', '--root', str(DATA),
        '--checkpoint', str(CHECKPOINT), '--out', str(OUT)], cwd=CODE, check=True)
finally:
    if OUT.exists():
        archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=NAME)
        with zipfile.ZipFile(archive) as z: assert z.testzip() is None
        import os
        os.chdir('/kaggle/working')
        print('Download:', archive)
        display(FileLink(Path(archive).name))


In [ ]:
record = json.loads((OUT / 'diagnostic.json').read_text())
assert record.get('status') == 'completed', 'Diagnostic did not finish.'
for row in json.loads((OUT / 'summary.json').read_text()):
    print(row['class_id'],row['scene'],row['mode'],
          'IoU:',round(100*row['target_iou'],2),'recall:',round(100*row['target_recall'],2))
print('These are selected TRAINING-scene diagnostics, not validation or test scores.')
